# Stage 2 — Data Cleaning & Wrangling
### Credit Card Fraud Detection · Tier A pipeline

**Purpose.** The data notes report zero nulls and 1,081 exact duplicate rows. There is no missing-value
strategy to design and no type coercion to repair. So this notebook does not perform cleaning theatre —
it does three things that actually matter:

1. **Verify, independently.** Re-derive every quality claim the data notes make. If a check contradicts
   the notes, the data wins and the contradiction is logged as a finding.
2. **Decide the duplicates** — on **leakage** grounds, not tidiness grounds. This is the consequential
   decision in the notebook and it gets a full argument.
3. **Derive the row-local columns** that `Time` and `Amount` support — and state, explicitly, the two
   families of features we are *refusing* to build and why.

**Governing plan:** `IMPLEMENTATION_PLAN.md` §3 · **Standard:** `DOCS/STRUCTURE.md` Stage 2

In [1]:
from __future__ import annotations

import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd

RANDOM_STATE = 42
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)

CWD = Path.cwd()
PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD
RAW_SRC = PROJECT_ROOT / "creditcard.csv" / "creditcard.csv"
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
REPORTS = PROJECT_ROOT / "reports"

V_COLS = [f"V{i}" for i in range(1, 29)]
DTYPE_CONTRACT = {"Time": "float64", "Amount": "float64", "Class": "int8"}
DTYPE_CONTRACT.update({c: "float64" for c in V_COLS})

# --- Re-verify the Stage 1 lineage hash before touching anything -------------------------------
def sha256_of(path: Path, chunk: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

lineage = pd.read_csv(DATA_RAW / "_lineage.csv")
expected_hash = lineage.loc[0, "sha256"]
actual_hash = sha256_of(RAW_SRC)
assert actual_hash == expected_hash, "SOURCE FILE CHANGED since ingestion. Re-run 01_ingestion.ipynb."
print(f"lineage hash verified : {actual_hash[:16]}...  (source unchanged since Stage 1)")

df = pd.read_csv(RAW_SRC, dtype=DTYPE_CONTRACT)
print(f"loaded                : {df.shape[0]:,} rows x {df.shape[1]} columns")

cleaning_log: list[dict] = []
def log(action, rows_affected, cols_affected, reason):
    cleaning_log.append({
        "step": len(cleaning_log) + 1, "action": action,
        "rows_affected": rows_affected, "cols_affected": cols_affected, "reason": reason,
    })

lineage hash verified : 76274b691b16a6c4...  (source unchanged since Stage 1)


loaded                : 284,807 rows x 31 columns


## 2.1 Verify, don't assume

Four independent checks against the claims in `DOCS/creditcard_dataset_analysis.md`. Each one is
re-derived from the file, not copied from the notes.

In [2]:
n_nulls = int(df.isna().sum().sum())
n_dupes = int(df.duplicated().sum())
time_sorted = bool(df["Time"].is_monotonic_increasing)
amount_neg = int((df["Amount"] < 0).sum())

verification = pd.DataFrame([
    {"claim": "zero missing values",        "notes_say": "0",     "we_observe": f"{n_nulls:,}",  "agrees": n_nulls == 0},
    {"claim": "exact duplicate rows",       "notes_say": "1,081", "we_observe": f"{n_dupes:,}",  "agrees": n_dupes == 1081},
    {"claim": "Time in transaction order",  "notes_say": "implied","we_observe": "sorted" if time_sorted else "UNSORTED", "agrees": time_sorted},
    {"claim": "Amount is non-negative",     "notes_say": "0 to 25,691", "we_observe": f"{amount_neg} negatives", "agrees": amount_neg == 0},
])
verification["verdict"] = np.where(verification["agrees"], "confirmed", "CONTRADICTS NOTES")
display(verification.drop(columns="agrees"))

if verification["agrees"].all():
    print("\nAll four documented claims are reproduced from the file itself. The notes are trustworthy.")
else:
    print("\nWARNING: the file contradicts the data notes above. The FILE governs; investigate before proceeding.")

,claim,notes_say,we_observe,verdict
0,zero missing values,0,0,confirmed
1,exact duplicate rows,"1,081","1,081",confirmed
2,Time in transaction order,implied,sorted,confirmed
3,Amount is non-negative,"0 to 25,691",0 negatives,confirmed



All four documented claims are reproduced from the file itself. The notes are trustworthy.


In [3]:
# Missingness strategy, documented per column as STRUCTURE.md requires — even though it is trivially "none".
null_by_col = df.isna().sum()
print("Missing-value strategy, per column:")
print(f"  all {len(df.columns)} columns: 0 nulls -> no imputation, no dropping, no strategy required.")
print(f"  max nulls in any single column: {int(null_by_col.max())}")
print()
print("This is not an oversight or a shortcut: the dataset is genuinely complete. Recording it")
print("explicitly is what distinguishes 'verified complete' from 'nobody checked'.")
log("verify missingness", 0, 31, "0 nulls across all 31 columns; no imputation strategy needed")

Missing-value strategy, per column:
  all 31 columns: 0 nulls -> no imputation, no dropping, no strategy required.
  max nulls in any single column: 0

This is not an oversight or a shortcut: the dataset is genuinely complete. Recording it
explicitly is what distinguishes 'verified complete' from 'nobody checked'.


## 2.2 The duplicate decision

1,081 rows are byte-identical to another row across all 31 columns. Two readings are possible and the
data cannot distinguish them:

- **Legitimate repeat charges** — a customer buys two identical coffees in the same second. Plausible
  for small amounts.
- **Ingestion artifacts** — the same transaction written twice.

Without a cardholder or merchant key, **these two are indistinguishable**. So the decision cannot be
made on "which is true"; it has to be made on **which error is more expensive**.

**It is a leakage question.** Duplicated rows that straddle a train/test boundary let the model score a
test row it has already memorised in training. That inflates test AUPRC for free, and does so most
severely on the positive class — where we have only 492 examples to begin with.

The asymmetry decides it:

| Choice | If we're wrong | Cost |
|---|---|---|
| **Drop** duplicates | They were genuine repeats | We lose ~0.4% of rows. The model sees marginally less data. |
| **Keep** duplicates | They were artifacts | Reported performance is inflated by memorisation, and we ship a number we cannot defend. |

**Decision: drop for the primary modelling table.** The counter-case is not asserted away — the
duplicates-retained table is preserved and the winning model is re-fit on it in notebook 05, with the
delta reported. If the headline number depends on this choice, the reader will see that it does.

In [4]:
dupe_mask = df.duplicated(keep="first")          # marks the redundant copies, keeps first occurrence
in_dupe_group = df.duplicated(keep=False)        # marks every row belonging to a duplicated group

dup_by_class = pd.DataFrame({
    "rows_in_class": df["Class"].value_counts().sort_index(),
    "redundant_copies": df.loc[dupe_mask, "Class"].value_counts().reindex([0, 1]).fillna(0).astype(int),
    "rows_in_a_dup_group": df.loc[in_dupe_group, "Class"].value_counts().reindex([0, 1]).fillna(0).astype(int),
})
dup_by_class.index = ["legitimate", "fraud"]
dup_by_class["pct_of_class_removed"] = (dup_by_class["redundant_copies"] / dup_by_class["rows_in_class"] * 100).round(3)
display(dup_by_class)

n_dup_fraud = int(dup_by_class.loc["fraud", "redundant_copies"])
n_dup_legit = int(dup_by_class.loc["legitimate", "redundant_copies"])
print(f"\n{n_dup_legit:,} redundant legitimate rows and {n_dup_fraud} redundant fraud rows.")
print(f"The fraud figure is the one that matters: {n_dup_fraud} of only 492 positives")
print(f"({n_dup_fraud / 492:.1%} of the entire positive class) are exact copies of another fraud row.")
print("Left in place, those are the rows most likely to be memorised and re-scored across the split.")

,rows_in_class,redundant_copies,rows_in_a_dup_group,pct_of_class_removed
legitimate,284315,1062,1822,0.374
fraud,492,19,32,3.862



1,062 redundant legitimate rows and 19 redundant fraud rows.
The fraud figure is the one that matters: 19 of only 492 positives
(3.9% of the entire positive class) are exact copies of another fraud row.
Left in place, those are the rows most likely to be memorised and re-scored across the split.


In [5]:
# Largest duplicate groups — is this a handful of rows repeated many times, or many pairs?
group_sizes = df[in_dupe_group].groupby(list(df.columns), observed=True).size()
print("Duplicate group-size distribution:")
print(group_sizes.value_counts().sort_index().rename("n_groups").to_frame().rename_axis("rows_per_group"))
print(f"\n{len(group_sizes):,} distinct duplicated records, accounting for {int(in_dupe_group.sum()):,} rows,")
print(f"of which {int(dupe_mask.sum()):,} are redundant copies to be removed.")
print("\nMostly pairs — consistent with either reading, which is exactly why the decision rests on")
print("leakage cost rather than on inferring intent.")

Duplicate group-size distribution:
                n_groups
rows_per_group          
2                    611
3                     66
4                     81
5                     10
6                      1
9                      2
18                     2

773 distinct duplicated records, accounting for 1,854 rows,
of which 1,081 are redundant copies to be removed.

Mostly pairs — consistent with either reading, which is exactly why the decision rests on
leakage cost rather than on inferring intent.


## 2.3 Derived columns — and the two families we refuse to build

Every column below is **row-local and fit-free**: computable from a single row (or the raw transaction
stream) with no reference to the target and no parameter estimated from data. That is what makes it safe
to compute here, before any split, without leaking.

| Column | Derivation | Serves |
|---|---|---|
| `hour_of_day` | `(Time % 86400) // 3600` | branch B2 — time-of-day fraud rate |
| `day` | `Time // 86400` → {0, 1} | the temporal-split key for notebook 05 |
| `amount_log` | `log1p(Amount)` | compresses a 25,691 : 22 skew |
| `amount_bucket` | 8 quantile bins | descriptive cross-tab vs `Class` |
| `sec_since_prev_txn` | `Time.diff()` on the sorted stream | branch B2 — global arrival gap |
| `is_dup_group_member` | full-row duplicate flag | audit trail for §2.2 |

### The refusals

**1. No feature is derived from V1–V28.** They are already principal components. Ratios, products or
interactions between them are algebraic recombinations of an orthogonal basis — they manufacture noise
and destroy the one property (near-orthogonality) that makes the basis trustworthy. `DOCS` §7 says this
plainly and we follow it.

**2. `amount_zscore` is deliberately skipped**, despite appearing in the data notes' suggestion list:

- computed **globally**, it is a monotone affine transform of `Amount` — it adds exactly zero information
  to any model, and to any rank-based metric it is literally the same column;
- computed **per class**, it uses the target to build the feature. That is textbook **target leakage**.

Naming the omission makes it a decision rather than an oversight.

**3. `sec_since_prev_txn` ships with a caveat attached.** The classic fraud signal is *per-card* velocity
— "this card has been used 5 times in 90 seconds". With no cardholder key, the best available proxy is
the gap to the previous transaction *anywhere in the portfolio*, which at ~5,900 transactions/hour is
mostly measuring global traffic rhythm, not any individual's behaviour. It is included so EDA can test
it, and it will be **dropped rather than defended** if it carries nothing.

In [6]:
# Computed on the full raw stream, in transaction order, BEFORE deduplication —
# the inter-arrival gap must reflect the real stream, not a stream with rows removed.
assert df["Time"].is_monotonic_increasing, "Time must be sorted for the inter-arrival gap to mean anything"

df["hour_of_day"] = ((df["Time"] % 86_400) // 3_600).astype("int8")
df["day"] = (df["Time"] // 86_400).astype("int8")
df["amount_log"] = np.log1p(df["Amount"])
df["sec_since_prev_txn"] = df["Time"].diff().fillna(0.0)
df["is_dup_group_member"] = in_dupe_group

qbins = pd.qcut(df["Amount"], q=8, duplicates="drop")
df["amount_bucket"] = qbins.cat.rename_categories(
    [f"Q{i+1}: {max(iv.left, 0):,.0f}-{iv.right:,.0f}" for i, iv in enumerate(qbins.cat.categories)]
)

for c, why in [("hour_of_day", "time-of-day fraud rate (branch B2)"),
               ("day", "temporal-split key for notebook 05"),
               ("amount_log", "compress 25,691:22 right-skew"),
               ("sec_since_prev_txn", "global inter-arrival gap (weak velocity proxy)"),
               ("is_dup_group_member", "audit trail for the duplicate decision"),
               ("amount_bucket", "descriptive cross-tab vs Class")]:
    log(f"derive column `{c}`", len(df), 1, why)

print(f"6 derived columns added -> {df.shape[1]} columns total\n")
display(df[["Time", "hour_of_day", "day", "Amount", "amount_log", "amount_bucket", "sec_since_prev_txn"]].head(4))
print(f"\nday split: {df['day'].value_counts().sort_index().to_dict()}  (Day 0 / Day 1 — the temporal-split key)")

6 derived columns added -> 37 columns total



,Time,hour_of_day,day,Amount,amount_log,amount_bucket,sec_since_prev_txn
0,0.0,0,0,149.62,5.014760,Q7: 77-165,0.0
1,0.0,0,0,2.69,1.305626,Q2: 2-6,0.0
2,1.0,0,0,378.66,5.939276,"Q8: 165-25,691",1.0
3,1.0,0,0,123.50,4.824306,Q7: 77-165,0.0



day split: {0: 144786, 1: 140021}  (Day 0 / Day 1 — the temporal-split key)


In [7]:
print("Sanity checks on the derived columns:\n")
print(f"  hour_of_day range        : [{df['hour_of_day'].min()}, {df['hour_of_day'].max()}]  (expect 0-23)")
print(f"  day values               : {sorted(df['day'].unique())}  (expect [0, 1])")
print(f"  amount_log vs Amount     : monotone = {bool((np.expm1(df['amount_log']).round(2) == df['Amount'].round(2)).all())}")
print(f"  sec_since_prev_txn median: {df['sec_since_prev_txn'].median():.0f}s, max {df['sec_since_prev_txn'].max():,.0f}s")
print(f"  ... share exactly 0s     : {(df['sec_since_prev_txn'] == 0).mean():.1%}")
print()
print("That last line is the caveat made concrete: at ~1.6 transactions per second portfolio-wide, a")
print("large share of gaps are zero. This column is measuring traffic rhythm, not individual velocity.")
print("EDA (notebook 03) decides whether it survives into the model.")

Sanity checks on the derived columns:

  hour_of_day range        : [0, 23]  (expect 0-23)
  day values               : [np.int8(0), np.int8(1)]  (expect [0, 1])
  amount_log vs Amount     : monotone = True
  sec_since_prev_txn median: 0s, max 32s
  ... share exactly 0s     : 56.3%

That last line is the caveat made concrete: at ~1.6 transactions per second portfolio-wide, a
large share of gaps are zero. This column is measuring traffic rhythm, not individual velocity.
EDA (notebook 03) decides whether it survives into the model.


## 2.4 Business-rule validation — including one rule that surfaces a real pattern

Standard range assertions, plus a check the data notes do not mention: **zero-amount transactions**.
A $0.00 authorisation is a recognised card-testing probe — a fraudster verifying a stolen card is live
before spending on it. Whether that pattern is present here is worth knowing before EDA.

In [8]:
n_zero_amt = int((df["Amount"] == 0).sum())
zero_fraud_rate = df.loc[df["Amount"] == 0, "Class"].mean()
overall_fraud_rate = df["Class"].mean()

rules = [
    ("Amount >= 0",                    bool((df["Amount"] >= 0).all()),                    f"min {df['Amount'].min():,.2f}"),
    ("Amount <= 25,691.16",            bool((df["Amount"] <= 25_691.16).all()),            f"max {df['Amount'].max():,.2f}"),
    ("Time within [0, 172792]",        bool(df["Time"].between(0, 172_792).all()),         f"[{df['Time'].min():,.0f}, {df['Time'].max():,.0f}]"),
    ("hour_of_day within [0, 23]",     bool(df["hour_of_day"].between(0, 23).all()),       "ok"),
    ("day within {0, 1}",              bool(df["day"].isin([0, 1]).all()),                 "ok"),
    ("Class within {0, 1}",            bool(df["Class"].isin([0, 1]).all()),               "ok"),
    ("no null introduced by derivation", int(df.isna().sum().sum()) == 0,                  f"{int(df.isna().sum().sum())} nulls"),
    ("sec_since_prev_txn >= 0",        bool((df["sec_since_prev_txn"] >= 0).all()),        f"min {df['sec_since_prev_txn'].min():.0f}"),
]
rules_df = pd.DataFrame(rules, columns=["rule", "passed", "observed"])
rules_df["result"] = np.where(rules_df["passed"], "PASS", "FAIL")
display(rules_df.drop(columns="passed"))
assert rules_df["passed"].all(), "business-rule validation failed"

print(f"\nZero-amount transactions: {n_zero_amt:,} rows ({n_zero_amt / len(df):.3%} of the file)")
print(f"  fraud rate among them  : {zero_fraud_rate:.3%}")
print(f"  fraud rate overall     : {overall_fraud_rate:.3%}")
print(f"  relative risk          : {zero_fraud_rate / overall_fraud_rate:,.1f}x the portfolio average")
print()
print("Retained, not treated as errors: a 0.00 authorisation is a legitimate record type (and a known")
print("card-testing probe). Quantified properly in notebook 04 — flagged here as a lead, not a finding.")
log("validate business rules", len(df), 8, "8/8 range and domain rules pass; zero-Amount rows retained as valid record type")

,rule,observed,result
0,Amount >= 0,min 0.00,PASS
1,"Amount <= 25,691.16","max 25,691.16",PASS
2,"Time within [0, 172792]","[0, 172,792]",PASS
3,"hour_of_day within [0, 23]",ok,PASS
4,"day within {0, 1}",ok,PASS
5,"Class within {0, 1}",ok,PASS
6,no null introduced by derivation,0 nulls,PASS
7,sec_since_prev_txn >= 0,min 0,PASS



Zero-amount transactions: 1,825 rows (0.641% of the file)
  fraud rate among them  : 1.479%
  fraud rate overall     : 0.173%
  relative risk          : 8.6x the portfolio average

Retained, not treated as errors: a 0.00 authorisation is a legitimate record type (and a known
card-testing probe). Quantified properly in notebook 04 — flagged here as a lead, not a finding.


## 2.5 Outliers — retained, with a documented reason

`DOCS/STRUCTURE.md`: *"never remove outliers without a documented reason."* Here the reason runs the
other way — removing them would delete the subject of the analysis.

- **`Amount` right tail** (max 25,691 against a median of 22) is genuine transaction behaviour. Handled
  by `log1p` plus robust scaling at model time, never deleted.
- **Extreme `V1`–`V28` values** are where fraud *lives*. The whole premise of the project is that fraud
  occupies unusual regions of this space. Trimming the tails would be deleting the signal and then
  reporting that no signal was found.

No outlier is removed anywhere in this pipeline. The quantification below is for the record.

In [9]:
def tail_share(s: pd.Series, k: float = 3.0) -> float:
    z = (s - s.mean()) / s.std()
    return float((z.abs() > k).mean())

tails = pd.DataFrame({
    "share_beyond_3sd_all": [tail_share(df[c]) for c in ["Amount"] + V_COLS[:6]],
    "share_beyond_3sd_fraud": [tail_share(df.loc[df["Class"] == 1, c]) for c in ["Amount"] + V_COLS[:6]],
}, index=["Amount"] + V_COLS[:6])
tails["fraud_enrichment"] = (tails["share_beyond_3sd_fraud"] / tails["share_beyond_3sd_all"]).replace([np.inf], np.nan)
display(tails.round(4))

print("Read the third column: on several components, fraud rows are many times more likely to sit in")
print("the extreme tail than the population is. Those tails are the signal. Trimming them would be")
print("deleting the finding and then reporting its absence.")
log("outlier treatment", 0, 0, "none removed; Amount tail is genuine behaviour, V-tails are the fraud signal itself")

,share_beyond_3sd_all,share_beyond_3sd_fraud,fraud_enrichment
Amount,0.0143,0.0224,1.5622
V1,0.0130,0.0305,2.3462
V2,0.0152,0.0183,1.2066
V3,0.0070,0.0203,2.9133
V4,0.0109,0.0000,0.0000
V5,0.0103,0.0122,1.1794
V6,0.0163,0.0163,0.9955


Read the third column: on several components, fraud rows are many times more likely to sit in
the extreme tail than the population is. Those tails are the signal. Trimming them would be
deleting the finding and then reporting its absence.


## 2.6 Feature engineering deferred to the modelling pipeline

Everything that requires a **fitted parameter** — scaling, cyclic encoding constants, any resampling —
is *not* done here. It belongs inside the `sklearn` `Pipeline` in notebook 05 so it fits on training
folds only.

This keeps Stage 2 a pure, leakage-free, deterministic transform: the same input file always produces
the same processed table, and nothing in that table has seen the target or the test set.

In [10]:
dedup = df.loc[~dupe_mask].reset_index(drop=True)
log("drop exact duplicate rows", int(dupe_mask.sum()), 31,
    "leakage control: identical rows straddling the train/test split are memorised, not learned; "
    f"{n_dup_fraud} of 492 positives affected. Duplicates-retained table preserved for sensitivity re-run.")

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
dedup.to_parquet(DATA_PROCESSED / "creditcard_clean.parquet", index=False)
df.to_parquet(DATA_PROCESSED / "creditcard_with_duplicates.parquet", index=False)

log_df = pd.DataFrame(cleaning_log)
log_df.to_csv(DATA_PROCESSED / "_cleaning_log.csv", index=False)

print(f"creditcard_clean.parquet            : {dedup.shape[0]:,} rows x {dedup.shape[1]} cols   <- THE modelling table")
print(f"creditcard_with_duplicates.parquet  : {df.shape[0]:,} rows x {df.shape[1]} cols   <- sensitivity re-run only")
print(f"\nrows removed: {int(dupe_mask.sum()):,}  ({dupe_mask.mean():.3%} of the file)")
print(f"fraud remaining: {int(dedup['Class'].sum())} of 492   |   fraud rate now {dedup['Class'].mean():.4%} (was {overall_fraud_rate:.4%})")
display(log_df)

creditcard_clean.parquet            : 283,726 rows x 37 cols   <- THE modelling table
creditcard_with_duplicates.parquet  : 284,807 rows x 37 cols   <- sensitivity re-run only

rows removed: 1,081  (0.380% of the file)
fraud remaining: 473 of 492   |   fraud rate now 0.1667% (was 0.1727%)


,step,action,rows_affected,cols_affected,reason
0,1,verify missingness,0,31,0 nulls across all 31 columns; no imputation s...
1,2,derive column `hour_of_day`,284807,1,time-of-day fraud rate (branch B2)
2,3,derive column `day`,284807,1,temporal-split key for notebook 05
3,4,derive column `amount_log`,284807,1,"compress 25,691:22 right-skew"
4,5,derive column `sec_since_prev_txn`,284807,1,global inter-arrival gap (weak velocity proxy)
5,6,derive column `is_dup_group_member`,284807,1,audit trail for the duplicate decision
6,7,derive column `amount_bucket`,284807,1,descriptive cross-tab vs Class
7,8,validate business rules,284807,8,8/8 range and domain rules pass; zero-Amount r...
8,9,outlier treatment,0,0,none removed; Amount tail is genuine behaviour...
9,10,drop exact duplicate rows,1081,31,leakage control: identical rows straddling the...


In [11]:
facts_path = REPORTS / "_key_figures.json"
figures = json.loads(facts_path.read_text()) if facts_path.exists() else {}
figures["stage_2_cleaning"] = {
    "n_nulls": n_nulls,
    "n_duplicate_rows_removed": int(dupe_mask.sum()),
    "n_duplicate_fraud_rows": n_dup_fraud,
    "dup_fraud_share_of_positives": float(n_dup_fraud / 492),
    "n_rows_clean": int(len(dedup)),
    "n_fraud_clean": int(dedup["Class"].sum()),
    "fraud_rate_clean": float(dedup["Class"].mean()),
    "imbalance_ratio_clean": float((dedup["Class"] == 0).sum() / dedup["Class"].sum()),
    "n_zero_amount": n_zero_amt,
    "zero_amount_fraud_rate": float(zero_fraud_rate),
    "zero_amount_relative_risk": float(zero_fraud_rate / overall_fraud_rate),
    "n_day0": int((dedup["day"] == 0).sum()),
    "n_day1": int((dedup["day"] == 1).sum()),
    "n_fraud_day0": int(dedup.loc[dedup["day"] == 0, "Class"].sum()),
    "n_fraud_day1": int(dedup.loc[dedup["day"] == 1, "Class"].sum()),
    "share_zero_arrival_gap": float((df["sec_since_prev_txn"] == 0).mean()),
}
facts_path.write_text(json.dumps(figures, indent=2))
print(json.dumps(figures["stage_2_cleaning"], indent=2))

{
  "n_nulls": 0,
  "n_duplicate_rows_removed": 1081,
  "n_duplicate_fraud_rows": 19,
  "dup_fraud_share_of_positives": 0.03861788617886179,
  "n_rows_clean": 283726,
  "n_fraud_clean": 473,
  "fraud_rate_clean": 0.001667101358352777,
  "imbalance_ratio_clean": 598.8435517970402,
  "n_zero_amount": 1825,
  "zero_amount_fraud_rate": 0.014794520547945205,
  "zero_amount_relative_risk": 8.564193117273637,
  "n_day0": 144236,
  "n_day1": 139490,
  "n_fraud_day0": 272,
  "n_fraud_day1": 201,
  "share_zero_arrival_gap": 0.5625423532427223
}


---

## Stage 2 — Gate Checklist (`DOCS/STRUCTURE.md`)

- [x] **Missing-value strategy documented per column** — 0 nulls across all 31 columns, independently re-derived; no imputation required and that is recorded rather than assumed
- [x] **Duplicates assessed and handled with justification** — 1,081 exact duplicates dropped on **leakage** grounds; the argument, the asymmetric-cost table, and the per-class breakdown are all in §2.2, and the duplicates-retained table is preserved for a sensitivity re-run
- [x] **All columns have correct dtypes** — enforced by contract at load; derived columns explicitly typed (`int8` for `hour_of_day` / `day`)
- [x] **Business-rule validations pass** — 8/8, plus a zero-Amount check that surfaced a lead for notebook 04
- [x] **Cleaning log exists** — `data/processed/_cleaning_log.csv`, one row per action with rows affected and reason
- [x] **Cleaned data saved** — `data/processed/creditcard_clean.parquet`

### Decisions a reviewer should be able to challenge

| Decision | Basis | How it's kept honest |
|---|---|---|
| Drop 1,081 duplicates | Leakage cost exceeds information cost; asymmetric-error table in §2.2 | Duplicates-retained table preserved; winning model re-fit on it in notebook 05 and the delta reported |
| Build no features from V1–V28 | They are already PCA outputs; recombination manufactures noise | Model uses the components as delivered |
| Skip `amount_zscore` | Global version is a monotone transform (adds nothing); per-class version is target leakage | Omission stated, not silent |
| Keep `sec_since_prev_txn` for now | Weak global proxy for per-card velocity, which is unbuildable here | EDA decides; dropped rather than defended if empty |
| Remove no outliers | The V-tails *are* the fraud signal | Tail-enrichment table quantifies why |

**Next:** `03_eda.ipynb` — hypothesis-driven exploration against the Stage 0 issue tree, one section per
branch, action titles and a written "So What" on every exhibit.